## Public Financials Retreival

** Steps:
1. Input company ticker and review company facts.
2. Get list of 10ks and 10Qs.
3. Get dataframes of the three financials statments for the input time horizon.
4. Export results to excel to be modeled.

In [1]:
from edgar import *
import pandas as pd

# Need to set your identity to use EDGAR API; input your email address
set_identity('purblindstocks@gmail.com')

c:\Users\cskur\OneDrive\Security Analysis\stock_report\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Set company ticker
company = Company('PFHO')
if company.not_found:
    print("Company not found")
company.docs

Company not found


╭──────────────────────────────────────────────────── Company ────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓  │
│  ┃                                        Company Class Documentation                                        ┃  │
│  ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛  │
│                                                                                                                 │
│                                                                                                                 │
│                                                    Overview                                                     │
│                                                                       

In [3]:
# Get company facts (my primary purpose here is to see their fiscal years and how many periods of financials we have available)
facts = company.get_facts()
facts

╭─────────────────────────── 📊 PACIFIC HEALTH CARE ORGANIZATION, INC. Financial Facts ───────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ ╭─────────────────────────────────────────── 📈 Summary Statistics ───────────────────────────────────────────╮ │
│ │                                                                                                             │ │
│ │   CIK               1138476                                                                                 │ │
│ │   Total Facts       12,255                                                                                  │ │
│ │   Unique Concepts   303                                                                                     │ │
│ │   Date Range        2011-08-12 to 2025-11-04                          

In [4]:
# Set the filing that you want to retrieve. In this case, we are getting 10 (or max) 10-K filings which contain the financial statements.

# Running this for PFHO and I want all 15 annual reports available, but there is a limit of 8 filings that can be stiched together
# So I have to break it up into two parts and then combine them later

head_annual_reports = company.get_filings(form="10-K").head(8)
tail_annual_reports = company.get_filings(form="10-K").tail(9)
quarterly_reports = company.get_filings(form="10-Q").head(1)
head_annual_reports, tail_annual_reports, quarterly_reports

(╭────────────────────────── Filings for PACIFIC HEALTH CARE ORGANIZATION INC [1138476] ───────────────────────────╮
 │                                                                                                                 │
 │                                                                              Filing                             │
 │    Form        Description                                                   Date         Accession Number      │
 │  ─────────────────────────────────────────────────────────────────────────────────────────────────────────────  │
 │    10-K        Annual report for public companies                            2025-03-19   0001185185-25-0001…   │
 │    10-K        Annual report for public companies                            2024-04-16   0001185185-24-0003…   │
 │    10-K        Annual report for public companies                            2023-03-31   0001185185-23-0003…   │
 │    10-K        Annual report for public companies            

In [16]:
# Extract the financial statements from the filings
new_multi_financials = MultiFinancials.extract(head_annual_reports)

# Assign the Income Statement, Balance Sheet, and Cash Flow Statement as objects
new_balance_sheet = new_multi_financials.balance_sheet()
new_income_statement = new_multi_financials.income_statement()
new_cashflow_statement = new_multi_financials.cashflow_statement()

# For the latter half of filings
old_multi_financials = MultiFinancials.extract(tail_annual_reports)

old_balance_sheet = old_multi_financials.balance_sheet()
old_income_statement = old_multi_financials.income_statement()
old_cashflow_statement = old_multi_financials.cashflow_statement()

# For the quarterly filings
quarterly_multi_financials = MultiFinancials.extract(quarterly_reports)

quarterly_balance_sheet = quarterly_multi_financials.balance_sheet()
quarterly_income_statement = quarterly_multi_financials.income_statement()
quarterly_cashflow_statement = quarterly_multi_financials.cashflow_statement()

No XBRL attachments found in filing Filing(company='PACIFIC HEALTH CARE ORGANIZATION INC', cik=1138476, form='10-K', filing_date='2011-03-31', accession_no='0001140377-11-000020')
No XBRL attachments found in filing Filing(company='PACIFIC HEALTH CARE ORGANIZATION INC', cik=1138476, form='10-K', filing_date='2010-03-31', accession_no='0001140377-10-000019')
No XBRL attachments found in filing Filing(company='PACIFIC HEALTH CARE ORGANIZATION INC', cik=1138476, form='10-K', filing_date='2010-03-31', accession_no='0001140377-10-000019')
No XBRL attachments found in filing Filing(company='PACIFIC HEALTH CARE ORGANIZATION INC', cik=1138476, form='10-K', filing_date='2009-03-31', accession_no='0001140377-09-000012')
No XBRL attachments found in filing Filing(company='PACIFIC HEALTH CARE ORGANIZATION INC', cik=1138476, form='10-K', filing_date='2009-03-31', accession_no='0001140377-09-000012')


In [17]:
# Print the object types from above
print(type(new_multi_financials))
print(type(new_balance_sheet))
print(type(new_income_statement))
print(type(new_cashflow_statement))

print(type(old_multi_financials))
print(type(old_balance_sheet))
print(type(old_income_statement))
print(type(old_cashflow_statement))

print(type(quarterly_multi_financials))
print(type(quarterly_balance_sheet))
print(type(quarterly_income_statement))
print(type(quarterly_cashflow_statement))

<class 'edgar.financials.MultiFinancials'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.financials.MultiFinancials'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.financials.MultiFinancials'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.xbrl.statements.StitchedStatement'>
<class 'edgar.xbrl.statements.StitchedStatement'>


In [18]:
# Print the newer multi-year statements
print("Recent Multi-Year Income Statement")
print(new_income_statement)
print("Recent Multi-Year Balance Sheet")
print(new_balance_sheet)
print("Recent Multi-Year Cash Flow Statement")
print(new_cashflow_statement)

Recent Multi-Year Income Statement


                                   CONSOLIDATED INCOME STATEMENT (8-Period View)                                   
                                                    Year Ended                                                     
                                                                                                                   
               FY Dec 31,   FY Dec 31,   FY Dec      FY Dec 31,   FY Dec      FY Dec 31,   FY Dec      FY Dec 31,  
               2024         2023         31, 2022    2021         31, 2020    2019         31, 2018    2017        
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
          R…   $6,065,390   $5,629,918   $5,744,9…   $5,403,110   $6,042,7…   $7,330,940   $6,796,9…   $6,505,527  
          S…   $2,754,394   $2,565,199   $2,689,8…   $2,740,806   $2,916,5…   $3,109,816   $2,462,2…   $2,299,698  
  and wages                                                             

                                    CONSOLIDATED BALANCE SHEET (8-Period View)                                     
                                                       As of                                                       
                                                                                                                   
               FY Dec 31,   FY Dec 31,   FY Dec      FY Dec 31,   FY Dec      FY Dec 31,   FY Dec      FY Dec 31,  
               2024         2023         31, 2022    2021         31, 2020    2019         31, 2018    2017        
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
      Cash     $2,070,476   $2,565,992   $2,036,4…   $10,085,3…   $9,498,4…   $8,104,164   $7,072,5…   $5,815,071  
  and Cash                                                                                                         
  Equivalen…                                                            

                               CONSOLIDATED STATEMENT OF CASH FLOWS (8-Period View)                                
                                                    Year Ended                                                     
                                                                                                                   
               FY Dec 31,   FY Dec 31,   FY Dec      FY Dec 31,   FY Dec      FY Dec 31,   FY Dec      FY Dec 31,  
               2024         2023         31, 2022    2021         31, 2020    2019         31, 2018    2017        
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
      Net        $883,584     $767,928    $492,886     $995,020    $549,570   $1,198,060   $1,359,7…     $964,405  
  Income                                                                                                           
        Dep…      $34,096      $37,527     $33,998      $48,887     $55,

In [ ]:
# Print the older multi-year statements
print("Older Multi-Year Income Statement")
print(old_income_statement)
print("Older Multi-Year Balance Sheet")
print(old_balance_sheet)
print("Older Multi-Year Cash Flow Statement")
print(old_cashflow_statement)

In [ ]:
# Print the quarterly multi-year statements
print("Quarterly Multi-Year Income Statement")
print(quarterly_income_statement)
print("Quarterly Multi-Year Balance Sheet")
print(quarterly_balance_sheet)
print("Quarterly Multi-Year Cash Flow Statement")
print(quarterly_cashflow_statement)

In [8]:
# Create dataframes from the multi-year statements

def create_df(statement):
    return statement.to_dataframe()

new_is_df, new_bs_df, new_cfs_df = [create_df(s) for s in (new_income_statement, new_balance_sheet, new_cashflow_statement)]
old_is_df, old_bs_df, old_cfs_df = [create_df(s) for s in (old_income_statement, old_balance_sheet, old_cashflow_statement)]
quar_is_df, quar_bs_df, quar_cfs_df = [create_df(s) for s in (quarterly_income_statement, quarterly_balance_sheet, quarterly_cashflow_statement)]

print(new_is_df.head())

                    label                               concept  2024-12-31  \
0                 Revenue                      us-gaap_Revenues   6065390.0   
1      Salaries and wages        us-gaap_LaborAndRelatedExpense   2754394.0   
2       Professional fees              us-gaap_ProfessionalFees    600915.0   
3               Insurance       us-gaap_GeneralInsuranceExpense    332856.0   
4  Outsource service fees  us-gaap_OtherCostAndExpenseOperating    699081.0   

   2023-12-31  2022-12-31  2021-12-31  2020-12-31  2019-12-31  2018-12-31  \
0   5629918.0   5744957.0   5403110.0   6042718.0   7330940.0   6796913.0   
1   2565199.0   2689842.0   2740806.0   2916576.0   3109816.0   2462253.0   
2    330236.0    314013.0    293936.0    295358.0    353394.0    347704.0   
3    311779.0    315919.0    321690.0    351122.0    328663.0    283844.0   
4    699770.0    610277.0    364951.0    449836.0    481695.0         NaN   

   2017-12-31  
0   6505527.0  
1   2299698.0  
2    352625.0 

In [9]:
# Merge the Income Statements (new and old)
# Use 'concept' column as the key to align rows, keep all columns from both
quar_is_new = pd.merge(quar_is_df, new_is_df, on='label', how='outer')
is_merged = pd.merge(quar_is_new, old_is_df, on='label', how='outer')

# Merge the Balance Sheets (new and old)
quar_bs_new = pd.merge(quar_bs_df, new_bs_df, on='label', how='outer')
bs_merged = pd.merge(quar_bs_new, old_bs_df, on='label', how='outer')

# Merge the Cash Flow Statements (new and old)
quar_cfs_new = pd.merge(quar_cfs_df, new_cfs_df, on='label', how='outer')
cfs_merged = pd.merge(quar_cfs_new, old_cfs_df, on='label', how='outer')

# Check the result
print("Merged Income Statement shape:", is_merged.shape)
print("Merged Balance Sheet shape:", bs_merged.shape)
print("Merged Cash Flow shape:", cfs_merged.shape)

print(is_merged)

Merged Income Statement shape: (47, 19)
Merged Balance Sheet shape: (60, 19)
Merged Cash Flow shape: (127, 19)
                                                label  \
0                                  Bad debt provision   
1                       Bad debt provision (recovery)   
2                                     Consulting fees   
3                     Cost of Goods and Services Sold   
4                                    Data maintenance   
5                                        Depreciation   
6                       Depreciation and amortization   
7                          Earnings Per Share (Basic)   
8                        Earnings Per Share (Diluted)   
9    Earnings per share amount (in Dollars per share)   
10                 General and Administrative Expense   
11                                           HCO fees   
12          Income (loss) before income tax provision   
13       Income Before Tax from Continuing Operations   
14                                

In [10]:
def remove_concept_column(df):
    cols_to_drop = [column for column in df.columns if 'concept' in column.lower()]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    return df

# Apply the function to all three merged dataframes
is_merged = remove_concept_column(is_merged)
bs_merged = remove_concept_column(bs_merged)
cfs_merged = remove_concept_column(cfs_merged)

print("Income Statement shape:", is_merged.shape)
print("Balance Sheet shape:", bs_merged.shape)
print("Cash Flow shape:", cfs_merged.shape)

# Show column names to verify
print("\nIncome Statement columns:", is_merged.columns.tolist())
print("\nBalance Sheet columns:", bs_merged.columns.tolist())
print("\nCash Flow Statement columns:", cfs_merged.columns.tolist())

# Check the data types of each dataframe
print("\n IS data types:", is_merged.dtypes)
print("\n BS data types:", bs_merged.dtypes)
print("\n CF data types:", cfs_merged.dtypes)

Income Statement shape: (47, 16)
Balance Sheet shape: (60, 16)
Cash Flow shape: (127, 16)

Income Statement columns: ['label', '2025-09-30', '2024-12-31', '2023-12-31', '2022-12-31', '2021-12-31', '2020-12-31', '2019-12-31', '2018-12-31', '2017-12-31', '2016-12-31', '2015-12-31', '2014-12-31', '2013-12-31', '2012-12-31', '2011-12-31']

Balance Sheet columns: ['label', '2025-09-30', '2024-12-31', '2023-12-31', '2022-12-31', '2021-12-31', '2020-12-31', '2019-12-31', '2018-12-31', '2017-12-31', '2016-12-31', '2015-12-31', '2014-12-31', '2013-12-31', '2012-12-31', '2011-12-31']

Cash Flow Statement columns: ['label', '2025-09-30', '2024-12-31', '2023-12-31', '2022-12-31', '2021-12-31', '2020-12-31', '2019-12-31', '2018-12-31', '2017-12-31', '2016-12-31', '2015-12-31', '2014-12-31', '2013-12-31', '2012-12-31', '2011-12-31']

 IS data types: label          object
2025-09-30    float64
2024-12-31    float64
2023-12-31    float64
2022-12-31    float64
2021-12-31    float64
2020-12-31    float6

In [11]:
# Some of the BS figures are still objects, need to convert to numeric
def convert_to_numeric(df):
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

bs_merged = convert_to_numeric(bs_merged)
is_merged = convert_to_numeric(is_merged)
cfs_merged = convert_to_numeric(cfs_merged)

# Check the result
print(bs_merged.dtypes)

label          object
2025-09-30    float64
2024-12-31    float64
2023-12-31    float64
2022-12-31    float64
2021-12-31    float64
2020-12-31    float64
2019-12-31    float64
2018-12-31    float64
2017-12-31    float64
2016-12-31    float64
2015-12-31    float64
2014-12-31    float64
2013-12-31    float64
2012-12-31    float64
2011-12-31    float64
dtype: object


In [14]:
import os
import shutil

# Copy the template file to the working directory
template_path = r"C:\Users\cskur\OneDrive\Security Analysis\_Template (Company) EVP and NCAV.xlsx"  # template location
output_path = rf"C:\Users\cskur\OneDrive\Security Analysis\Company Analysis\PFHO\Pacific Health Care Organization - EVP and NCAV.xlsx"

# Copy the template
shutil.copy(template_path, output_path)
print(f"Template copied to {output_path}")

# Open the existing file and write to specific sheets (without deleting other tabs)
with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    is_merged.to_excel(writer, sheet_name='Income Statement', index=False)
    bs_merged.to_excel(writer, sheet_name='Balance Sheet', index=False)
    cfs_merged.to_excel(writer, sheet_name='Cash Flow', index=False)

print("Financial statements written to Excel file")

Template copied to C:\Users\cskur\OneDrive\Security Analysis\Company Analysis\PFHO\Pacific Health Care Organization - EVP and NCAV.xlsx


c:\Users\cskur\OneDrive\Security Analysis\stock_report\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:313: UserWarning: Failed to load a conditional formatting rule. It will be discarded. Cause: expected <class 'float'>
  warn(msg)


Financial statements written to Excel file
